In [2]:
import pandas as pd
import numpy as np

In [3]:
def drop_cols(df, cols=[]):
    #Cleans redundant columns (if there's no values for any datasets in the column, remove that specific column)
    print(f"Original columns: {len(df.columns)}")
    #Further refines the column field by removing repeat columns 
    filtered_df = df.drop(columns=cols)
    print(f"Dropped columns: {len(cols)}")
    print(f"Remaining columns: {len(filtered_df.columns)}")
    return filtered_df


## Filter to all spectroscopy done on gases 

In [4]:
#Script to trim down the 1.3GB dataset by filtering for gas only states
t = pd.read_csv(r"/home/mmalik/Documents/Projects/CscSpring26/CSC_4444/CSC4444_Project/Organic-Compound-Classifier/data/IR spectroscopy data/NIST_IR_Spectroscopy.csv",low_memory=False) #Used relative path based on my local machine. Change as needed

gas_df= t[t['state'].str.contains('gas',case=False,na=False)]

print(f"Original rows:{len(t)}")
print(f"Gas rows: {len(gas_df)}")

#Cleans redundant columns (if there's no values for any datasets in the column, remove that specific column)
gas_df_columnCleaned = gas_df.dropna(axis=1, how='all')
redundant_axis_dropped = ['maxx', 'minx', 'maxy', 'miny']
gas_df_columnCleaned = drop_cols(gas_df_columnCleaned, redundant_axis_dropped)


Original rows:19213
Gas rows: 9989
Original columns: 55
Dropped columns: 4
Remaining columns: 51


## Feature selection

### what is being dropped
* mostly null cols
* columns with low variance / unique to each row(id)
* columns that can be derived from smiles (melting point, name of compound, molecular formula, etc)
* cols that have high correlation with another

In [30]:
post_feature_selection_rows = ["filename","smiles", "xunits", "yunits", "xfactor",  "yfactor", "x_coords", "y_coords"]
selected_df = gas_df_columnCleaned[post_feature_selection_rows]


## Standardize x,y coordinate data

### convert all x units to 1/cm

In [43]:
print("number of rows before filtering out x units, ",selected_df.count())
filtered_x_unit_df=selected_df[selected_df["xunits"].isin(["1/CM", "cm-1"])]
print("number of rows after, ",filtered_x_unit_df.count())

number of rows before filtering out x units,  filename    9989
smiles      9989
xunits      9989
yunits      9989
xfactor     9989
yfactor     9989
x_coords    9989
y_coords    9989
dtype: int64
number of rows after,  filename    9972
smiles      9972
xunits      9972
yunits      9972
xfactor     9972
yfactor     9972
x_coords    9972
y_coords    9972
dtype: int64


### normalize x coords 

In [ ]:
import ast
filtered_x_unit_df["x_coords"] = filtered_x_unit_df["x_coords"].apply(ast.literal_eval)
filtered_x_unit_df["x_coords"] = filtered_x_unit_df.apply(
    lambda row: [coord * float(row["xfactor"]) for coord in row["x_coords"]],
    axis=1
)

,filename,smiles,xunits,yunits,xfactor,yfactor,x_coords,y_coords
1381,B6010491_IR_0.jdx,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,1/CM,TRANSMITTANCE,1.0,1.000000,"[451.282, 452.1367016, 452.9914032, 453.846104...","[0.932, 0.93, 0.929, 0.928, 0.925, 0.92, 0.917..."
1382,B6010491_IR_1.jdx,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,1/CM,TRANSMITTANCE,1.0,1.000000,"[456.596, 457.8726036, 459.1492072, 460.425810...","[0.91, 0.908, 0.904, 0.89, 0.8621, 0.8361, 0.7..."
1383,B6010497_IR_0.jdx,CC(=O)N[C@@H]1[C@@H](O)C[C@](O)(O[C@H]1[C@H](O...,1/CM,TRANSMITTANCE,1.0,1.000000,"[403.266, 404.5222826, 405.7785652, 407.034847...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1389,C100005_IR_0.jdx,[O-][N+](=O)c1ccc(Cl)cc1,1/CM,ABSORBANCE,1.0,0.000127,"[450.0, 454.0, 458.0, 462.0, 466.0, 470.0, 474...","[0.025064226, 0.038735622, 0.039495144, 0.0196..."
1391,C100016_IR_0.jdx,Nc1ccc(cc1)[N+]([O-])=O,1/CM,ABSORBANCE,1.0,0.001000,"[551.688, 555.546, 559.404, 563.262, 567.12, 5...","[0.00505, 0.00536, 0.00556, 0.00599, 0.00634, ..."


In [ ]:
#drop x factor
filtered_x_unit_df = drop_cols(filtered_x_unit_df, ["xfactor"])

Original columns: 7


KeyError: "['xfactor'] not found in axis"

### normalize all y units (transmitance, etc..) to absorbance

In [9]:
import ast
#for all cols with y unit equal to transmitance for all y in y_coords set their value to -np.log10(y_raw * yfactor) then set cols y unit converted to x

# for all cols with y unit equal to 

def convert_micromol(row):
    y_raw = ast.literal_eval(row['y_coords'])
    return [val * float(row['yfactor']) for val in y_raw]

def convert_transmittance(row):
    y_raw = ast.literal_eval(row['y_coords'])
    return [-np.log10(val * float(row['yfactor'])) for val in y_raw]

transmittance_mask = df_y_normalized['yunits'] == 'TRANSMITTANCE'
df_y_normalized.loc[transmittance_mask, 'y_coords'] = (
    df_y_normalized[transmittance_mask].apply(convert_transmittance, axis=1)
)
df_y_normalized.loc[transmittance_mask, 'yunits'] = 'ABSORBANCE'

micromol_mask = df_y_normalized['yunits'] == '(micromol/mol)-1m-1 (base 10)'
df_y_normalized.loc[micromol_mask, 'y_coords'] = (
    df_y_normalized[micromol_mask].apply(convert_micromol, axis=1)
)
df_y_normalized.loc[micromol_mask, 'yunits'] = 'ABSORBANCE'
df_y_normalized.head()

/tmp/ipykernel_17677/2647728715.py:12: RuntimeWarning: divide by zero encountered in log10
  return [-np.log10(val * float(row['yfactor'])) for val in y_raw]


,filename,title,jcamp-dx,data type,class,origin,owner,date,names,molform,...,sample description,pressure,temperature,smiles,smiles_method,smiles_query,x_coords,y_coords,x_normalized,y_normalized
6298,C17700093_IR_0.jdx,"1,2,3-Trichloro-4-nitrobenzene",4.24,INFRARED SPECTRUM,NaN,Sadtler Research Labs Under US-EPA Contract,NIST Standard Reference Data Program Collectio...,NaN,NaN,C 6 H 2 Cl 3 N O 2,...,NaN,NaN,NaN,[O-][N+](=O)c1ccc(Cl)c(Cl)c1Cl,cirpy:cas registry no,17700-09-3,"[450.0, 454.0, 458.0, 462.0, 466.0, 470.0, 474...","[0.027312638, 0.022792931, 0.026194861, 0.0, 0...",None,None
2258,C106423_IR_9.jdx,"1,4-Dimethylbenzene",4.24,INFRARED SPECTRUM,IUPAC A,"NIST, Analytical Chemistry Division, 301-975-3108",COPYRIGHT (C) 2001 by the U.S. Secretary of Co...,01/06/01,NaN,"1,4-( C H3)2 C6 H4",...,p-Xylene Primary Gas Standard,101.3 Pa,23 C,Cc1ccc(C)cc1,cirpy:cas registry no,106-42-3,"[575.05, 575.0999999999999, 575.15, 575.2, 575...","[-3.762114185552656e-18, 2.982411048465615e-18...",None,None
6425,C18189087_IR_0.jdx,"Anthranilic acid, 5-chloro-, butyl ester",4.24,INFRARED SPECTRUM,NaN,Sadtler Research Labs Under US-EPA Contract,NIST Standard Reference Data Program Collectio...,NaN,NaN,C 11 H 14 Cl N O 2,...,NaN,NaN,NaN,CCCCOC(=O)c1cc(Cl)ccc1N,cirpy:cas registry no,18189-08-7,"[450.0, 454.0, 458.0, 462.0, 466.0, 470.0, 474...","[0.0, 0.002069235, 0.003034878, 0.007035399, 0...",None,None
1519,C10045655_IR_0.jdx,"1H-Imidazole-2-carboxaldehyde, 1-(phenylmethyl)-",4.24,INFRARED SPECTRUM,NaN,Sadtler Research Labs Under US-EPA Contract,NIST Standard Reference Data Program Collectio...,NaN,NaN,C 11 H 10 N 2 O,...,NaN,NaN,NaN,O=Cc1nccn1Cc2ccccc2,cirpy:cas registry no,10045-65-5,"[450.0, 454.0, 458.0, 462.0, 466.0, 470.0, 474...","[0.008605135, 0.01078616, 0.0108258149999999, ...",None,None
6210,C17452083_IR_0.jdx,1-(3-Trifluoromethylphenyl)imidazoline-2-thione,4.24,INFRARED SPECTRUM,NaN,EPA-IR VAPOR PHASE LIBRARY,SRD/NIST Collection (C) 2018 copyright by the ...,NaN,NaN,C 10 H 7 F 3 N 2 S,...,NaN,NaN,NaN,FC(F)(F)c1cccc(c1)N2C=CNC2=S,cirpy:cas registry no,17452-08-3,"[549.759, 551.688, 553.617, 555.546, 557.475, ...","[0.009375, 0.008895, 0.008475, 0.008215, 0.008...",None,None


### trim range of y values to cut off outlyers

In [10]:

#convert y_coords to eal list of ints

#valud range of absorbance vlaues is 

### convert all x to 1/cm

In [11]:
#keep cols 
